<a href="https://sigmoidal.ai">
  <img src="https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/_assets/logo_sigmoidal.png" alt="Sigmoidal" width="300">
</a>

# Transfer Learning com PyTorch

**Autor:** Carlos Melo — [sigmoidal.ai](https://sigmoidal.ai)

## Bloco 1 — Carregando o Oxford Flowers 102

Vamos usar o dataset Oxford Flowers 102, que contém imagens de 102 espécies de flores. Com apenas 1020 imagens de treino, é um cenário ideal para transfer learning.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

A ResNet18 espera imagens 224x224. Aplicamos data augmentation no treino e apenas redimensionamento no teste.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = torchvision.datasets.Flowers102(
    root="./data", split="train", download=True, transform=train_transform
)

val_dataset = torchvision.datasets.Flowers102(
    root="./data", split="val", download=True, transform=test_transform
)

test_dataset = torchvision.datasets.Flowers102(
    root="./data", split="test", download=True, transform=test_transform
)

print("Treino:", len(train_dataset), "imagens")
print("Validacao:", len(val_dataset), "imagens")
print("Teste:", len(test_dataset), "imagens")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Vamos visualizar algumas imagens do dataset, desnormalizando com a média e desvio padrão do ImageNet.

In [ ]:
def desnormalizar(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img * std + mean
    img = torch.clamp(img, 0, 1)
    return img

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    img = desnormalizar(images[i])
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f"Classe {labels[i].item()}")
    ax.axis("off")
plt.suptitle("Exemplos do Oxford Flowers 102")
plt.tight_layout()
plt.show()

## Bloco 2 — Explorando a ResNet18

A ResNet18 é uma rede de 18 camadas com conexões residuais, pré-treinada no ImageNet (1000 classes). Vamos carregá-la e entender sua estrutura antes de adaptá-la.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

model_original = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model_original.eval()

print(model_original)

O backbone extrai features e a camada `fc` classifica. Vamos inspecionar os componentes principais.

In [ ]:
# Camada inicial
print("Entrada: Conv2d + BN + ReLU + MaxPool")
print(model_original.conv1)
print(model_original.bn1)
print()

# Blocos residuais
print("Bloco layer1:", model_original.layer1)
print()

# Cabeca de classificacao
print("Average Pool:", model_original.avgpool)
print("FC (classificador):", model_original.fc)
print(f"  Entrada: {model_original.fc.in_features} features")
print(f"  Saida: {model_original.fc.out_features} classes (ImageNet)")

Antes de adaptar a rede, vamos ver como ela classifica uma flor usando as 1000 classes do ImageNet.

In [ ]:
import json
import urllib.request

# Baixar nomes das classes do ImageNet
url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
urllib.request.urlretrieve(url, "imagenet_labels.json")

with open("imagenet_labels.json") as f:
    imagenet_labels = json.load(f)

print(f"Total de classes ImageNet: {len(imagenet_labels)}")
print("Primeiras 5:", imagenet_labels[:5])

In [ ]:
# Pegar uma imagem de flor do dataset
img, label = images[0], labels[0]

model_original = model_original.to(device)
img_gpu = img.unsqueeze(0).to(device)

with torch.no_grad():
    output = model_original(img_gpu)
    probs = torch.softmax(output, dim=1)
    top5_probs, top5_idx = probs.topk(5)

print("Predicoes da ResNet18 (ImageNet) para uma flor:")
print()
for i in range(5):
    idx = top5_idx[0][i].item()
    prob = top5_probs[0][i].item()
    print(f"  {imagenet_labels[idx]:>30s}: {prob:.1%}")

plt.imshow(desnormalizar(img).permute(1, 2, 0).numpy())
plt.title(f"Label real: classe {label.item()}")
plt.axis("off")
plt.show()

## Bloco 3 — Feature Extraction

Congelamos todo o backbone e treinamos apenas uma nova camada de classificação. O backbone funciona como extrator de features fixo.

In [ ]:
model_fe = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Congelar todos os parametros do backbone
for param in model_fe.parameters():
    param.requires_grad = False

# Trocar a camada fc para 102 classes
num_features = model_fe.fc.in_features
model_fe.fc = nn.Linear(num_features, 102)

# Somente a nova fc tem gradientes
print("Parametros treinaveis:")
for name, param in model_fe.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

model_fe = model_fe.to(device)

Função de treino genérica que vamos reutilizar nos dois experimentos.

In [ ]:
def treinar(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    train_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        # Treino
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)

        # Validacao
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        val_accuracies.append(val_acc)

        print(f"Epoca {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f} - Val Acc: {val_acc:.1f}%")

    return train_losses, val_accuracies

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer_fe = optim.Adam(model_fe.fc.parameters(), lr=0.001)

print("Treinando com feature extraction (backbone congelado)")
print()
fe_losses, fe_accs = treinar(model_fe, train_loader, val_loader, criterion, optimizer_fe, num_epochs=15)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(fe_losses)
plt.xlabel("Epoca")
plt.ylabel("Loss")
plt.title("Loss - Feature Extraction")

plt.subplot(1, 2, 2)
plt.plot(fe_accs)
plt.xlabel("Epoca")
plt.ylabel("Acuracia (%)")
plt.title("Val Accuracy - Feature Extraction")

plt.tight_layout()
plt.show()

## Bloco 4 — Fine-Tuning

Agora descongelamos a `layer4` (ultimo bloco residual) e treinamos junto com a cabeça, usando learning rates diferenciados para não destruir os pesos pré-treinados.

In [ ]:
model_ft = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Congelar tudo primeiro
for param in model_ft.parameters():
    param.requires_grad = False

# Descongelar layer4
for param in model_ft.layer4.parameters():
    param.requires_grad = True

# Trocar a fc
model_ft.fc = nn.Linear(model_ft.fc.in_features, 102)

# Verificar o que esta treinavel
total_params = sum(p.numel() for p in model_ft.parameters())
trainable_params = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Parametros totais: {total_params:,}")
print(f"Parametros treinaveis: {trainable_params:,}")
print(f"Percentual treinavel: {100*trainable_params/total_params:.1f}%")

model_ft = model_ft.to(device)

Learning rate menor para o backbone (pesos já bons) e maior para a cabeça nova.

In [ ]:
optimizer_ft = optim.Adam([
    {"params": model_ft.layer4.parameters(), "lr": 1e-4},
    {"params": model_ft.fc.parameters(), "lr": 1e-3},
])

print("Treinando com fine-tuning (layer4 + fc)")
print()
ft_losses, ft_accs = treinar(model_ft, train_loader, val_loader, criterion, optimizer_ft, num_epochs=15)

### Comparando as duas abordagens

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(fe_losses, label="Feature Extraction")
plt.plot(ft_losses, label="Fine-Tuning")
plt.xlabel("Epoca")
plt.ylabel("Loss")
plt.title("Loss de treino")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(fe_accs, label="Feature Extraction")
plt.plot(ft_accs, label="Fine-Tuning")
plt.xlabel("Epoca")
plt.ylabel("Acuracia (%)")
plt.title("Acuracia na validacao")
plt.legend()

plt.tight_layout()
plt.show()

print(f"Melhor acuracia Feature Extraction: {max(fe_accs):.1f}%")
print(f"Melhor acuracia Fine-Tuning: {max(ft_accs):.1f}%")

### Acurácia no conjunto de teste

In [ ]:
model_ft.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_ft(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f"Acuracia no teste (Fine-Tuning): {test_acc:.1f}%")